# 使用 seegops 读取 0807 华山 Grip Flight 数据

## tl;dr

本笔记本用今天建立的 `seegops` 工具读取两组BCI2000主流与任务流，检查形状、采样率、状态字段、基础QC和记录起点。256通道主流的文件头声明 `SignalGeneratorADC`，因此只能作为格式和算子测试数据，不能作为真实sEEG进行生理解释。

## Context & Methods

### Key Assumptions

- BCI2000文件均通过内存映射读取，不复制完整数据到RAM。
- 主流与任务流是两个独立时钟；`StorageTime`只用于粗起点诊断。
- 本笔记本不把任务事件映射到256通道主流，因为尚无共享TTL或漂移估计。
- `EventTable`在本笔记本中只建立于任务流自己的时钟。

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

WORKSPACE = Path('/Users/heting/Documents/seeg-operator-workbench')
DATA_ROOT = Path('/Users/heting/Documents/readGripData/0807华山grip flight')
sys.path.insert(0, str(WORKSPACE / 'src'))

from seegops.io import read_bci2000
from seegops.qc import basic_qc
from seegops.sync import align_by_storage_time, rising_edges
from seegops import EventTable

assert DATA_ROOT.exists(), DATA_ROOT
print('Workspace:', WORKSPACE)
print('Data root:', DATA_ROOT)
print('seegops import: OK')

Workspace: /Users/heting/Documents/seeg-operator-workbench
Data root: /Users/heting/Documents/readGripData/0807华山grip flight
seegops import: OK


## Data

### 1. 读取四个文件

In [2]:
recordings = {}
for run in ('09', '11'):
    recordings[(run, 'main')] = read_bci2000(DATA_ROOT / f'testS001R{run}.dat.larkcache')
    recordings[(run, 'task')] = read_bci2000(
        DATA_ROOT / f'testS001R{run}_1.dat',
        channel_names=['GripForce1'],
    )

summary_rows = []
for (run, stream), recording in recordings.items():
    summary = recording.summary()
    summary_rows.append({
        'run': run,
        'stream': stream,
        'channels': summary['shape'][0],
        'samples': summary['shape'][1],
        'sampling_rate_hz': summary['sampling_rate_hz'],
        'duration_s': summary['duration_s'],
        'dtype': summary['dtype'],
        'unit': summary['unit'],
        'storage_time': summary['storage_time'],
    })
summary_df = pd.DataFrame(summary_rows).sort_values(['run', 'stream'])
print(summary_df.to_string(index=False))

run stream  channels  samples  sampling_rate_hz  duration_s   dtype unit               storage_time
 09   main       256   286800            2000.0      143.40   int16  ADU 2026-08-07T16:42:27.213000
 09   task         1    31488             256.0      123.00 float32  ADU 2026-08-07T16:42:27.045000
 11   main       256   314320            2000.0      157.16   int16  ADU 2026-08-07T17:46:01.451000
 11   task         1    38208             256.0      149.25 float32  ADU 2026-08-07T17:46:01.316000


### 2. 有界预览：主流和握力流的前几个采样点

In [3]:
main09 = recordings[('09', 'main')]
task09 = recordings[('09', 'task')]

main_preview = pd.DataFrame(
    np.asarray(main09.signal.data[:4, :10]).T,
    columns=main09.signal.coords['channel'][:4],
)
task_preview = pd.DataFrame({
    'time_s': task09.signal.coords['time'][:10],
    'GripForce1_ADU': np.asarray(task09.signal.data[0, :10]),
    'GamePhase': task09.state('GamePhase')[:10],
    'GripForceNormalized_state': task09.state('GripForceNormalized')[:10],
})
print('Run 09 main stream, first 4 channels × 10 samples:')
print(main_preview.to_string(index=False))
print('\nRun 09 task stream, first 10 samples:')
print(task_preview.to_string(index=False))

Run 09 main stream, first 4 channels × 10 samples:
 CH001  CH002  CH003  CH004
     2     54    -96    -24
  -133    -41   -109     -2
   -93    -54    -61     15
    27    148    102     12
   136     98    139    -31
   139    -73    105   -134
   -56     63     56    -16
    52     40     49      5
    75    -10    140    -88
   -82     42     85    -17

Run 09 task stream, first 10 samples:
  time_s  GripForce1_ADU  GamePhase  GripForceNormalized_state
0.000000           0.811          0                          0
0.003906           0.811          0                          0
0.007812           0.811          0                          0
0.011719           0.811          0                          0
0.015625           0.811          0                          0
0.019531           0.811          0                          0
0.023438           0.811          0                          0
0.027344           0.811          0                          0
0.031250           0.811          0

## Results

### 3. 任务状态和事件读取

In [4]:
state_rows = []
feedback_tables = {}
for run in ('09', '11'):
    task = recordings[(run, 'task')]
    phase = task.state('GamePhase')
    feedback = task.state('Feedback')
    collision = task.state('Collision')
    feedback_events = EventTable.from_state_edges(
        feedback,
        sampling_rate=task.signal.sampling_rate,
        event_type='feedback',
        id_prefix=f'run-{run}-feedback',
    )
    feedback_tables[run] = feedback_events
    state_rows.append({
        'run': run,
        'GamePhase_values': sorted(set(map(int, phase))),
        'feedback_onsets_s': [round(e.onset_s, 3) for e in feedback_events.events],
        'collision_onsets_s': (rising_edges(collision) / task.signal.sampling_rate).round(3).tolist(),
    })
print(pd.DataFrame(state_rows).to_string(index=False))
print('\nRun 09 feedback EventTable records:')
print(pd.DataFrame(feedback_tables['09'].as_records()).to_string(index=False))

run   GamePhase_values            feedback_onsets_s collision_onsets_s
 09 [0, 1, 2, 4, 5, 6]        [2.125, 50.125, 77.0]             [74.0]
 11 [0, 1, 2, 4, 5, 6] [2.125, 7.25, 55.25, 103.25]             [4.25]

Run 09 feedback EventTable records:
           event_id event_type  onset_s  duration_s trial_id value  valid  source_sample            source
run-09-feedback-001   feedback    2.125         0.0     None  None   True            544 state_rising_edge
run-09-feedback-002   feedback   50.125         0.0     None  None   True          12832 state_rising_edge
run-09-feedback-003   feedback   77.000         0.0     None  None   True          19712 state_rising_edge


### 4. 基础QC与模拟信号警告

In [5]:
qc_rows = []
for run in ('09', '11'):
    qc = basic_qc(recordings[(run, 'main')].signal, seconds=10.0)
    qc_rows.append({
        'run': run,
        'finite_fraction': qc['finite_fraction'],
        'constant_channels': len(qc['constant_channels']),
        'rms_median_raw': round(qc['rms_median_raw'], 3),
        'rms_cv': round(qc['rms_coefficient_of_variation'], 5),
        'peak_to_peak_median_raw': qc['peak_to_peak_median_raw'],
        'warnings': ' | '.join(qc['warnings']),
    })
qc_df = pd.DataFrame(qc_rows)
print(qc_df.to_string(index=False))

run  finite_fraction  constant_channels  rms_median_raw  rms_cv  peak_to_peak_median_raw                                                                                                                                                                                                   warnings
 09              1.0                  0          86.157 0.00334                    298.0 BCI2000 header declares SignalGeneratorADC; treat as synthetic until acquisition provenance proves otherwise. | Channel RMS values are unusually homogeneous; inspect for generated or duplicated signals.
 11              1.0                  0          86.134 0.00308                    298.0 BCI2000 header declares SignalGeneratorADC; treat as synthetic until acquisition provenance proves otherwise. | Channel RMS values are unusually homogeneous; inspect for generated or duplicated signals.


### 5. 两条流的粗起点诊断

In [6]:
alignment_rows = []
for run in ('09', '11'):
    alignment = align_by_storage_time(
        recordings[(run, 'main')],
        recordings[(run, 'task')],
    )
    alignment_rows.append({
        'run': run,
        'task_start_minus_main_start_s': alignment.offset_s,
        'metadata_resolution_bound_s': alignment.uncertainty_s,
        'method': alignment.method,
    })
print(pd.DataFrame(alignment_rows).to_string(index=False))
print('\n注意：这不能证明两条流采样锁定，也不能估计时钟漂移。')

run  task_start_minus_main_start_s  metadata_resolution_bound_s                       method
 09                         -0.168                     0.003906 BCI2000 StorageTime metadata
 11                         -0.135                     0.003906 BCI2000 StorageTime metadata

注意：这不能证明两条流采样锁定，也不能估计时钟漂移。


## Takeaways

- 四个文件均已通过 `seegops.io.read_bci2000` 成功读取，且使用内存映射。
- Run 09/11主流分别为256通道、2000 Hz、143.40/157.16秒；任务流为单通道、256 Hz、123.00/149.25秒。
- 任务流的BCI2000 states可解码，并已把Feedback上升沿转换成EventTable。
- 256通道主流触发 `SignalGeneratorADC` 和异常均一RMS警告，应视为模拟信号。
- 在获得共享同步脉冲或时钟映射之前，不应把任务事件直接映射到主流生成epoch。